In [ ]:
// First cell initializes Spark Context, no need to create one in code

// Imports
import org.apache.spark.{SparkContext, SparkConf}
import org.apache.spark.rdd.RDD
import org.apache.hadoop.fs._

import scala.util.parsing.json._


In [ ]:
// File locations
val dev_set_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json"
val full_set_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviewscombined.json"

// I don't like this hardcoded approach, but relative or local paths wouldn't work
val USER = "e12233000"
val stopwords_path = "hdfs:///user/"+USER+"/data/stopwords.txt"

val VERBOSE: Boolean = false
val DEVEL: Boolean = true

In [ ]:
// Helper functions

// Print first k elements of an RDD, k default 2
// T is template parameter
def pprint[T](rdd: org.apache.spark.rdd.RDD[T], k: Int = 2): Unit = {
    if (VERBOSE == true) {
        rdd.take(k).foreach(println)
    }
    
}

// Compute chi-squares by N, A, B, C
def chi_square(N: Int, A: Int, B: Int, C: Int): Double = {
    val D = N - A - B - C;
    val num = N * Math.pow((A*D - B*C), 2);
    val den = (A + C) * (B + D) * (A + B) * (C + D);
    if (den == 0) return 0.0;
    else return num / den;
}

In [ ]:
// Load dataset into an RDD
val dataset = if (DEVEL) sc.textFile(dev_set_path) else sc.textFile(full_set_path)
// Load stopwords into a Scala set
val stopwords = sc.textFile(stopwords_path).collect().toSet

In [ ]:
// Explore how the lines look
pprint(dataset)

In [ ]:
// Parsing JSON line
val categoryReview = dataset.map(line => {
  val json:Option[Any] = JSON.parseFull(line)
  val map:Map[String,Any] = json.get.asInstanceOf[Map[String, Any]]
  val category = map("category").toString
  val review = map("reviewText").toString
  (category, review)
})

In [ ]:
pprint(categoryReview,5)

In [ ]:
// Lowercase, tokenize, remove duplicates and filter stopwords
var categoryWord = categoryReview.flatMap{ case (category, text) =>
    val words = text.toLowerCase.split("[ \t0-9\\(\\)\\[\\]\\{\\}\\.\\!\\?,;:\\+=\\-_\"'`~#@&%*%€$§\\\\/|<>^]+").distinct
    words.map(word => (category, word))
}
categoryWord = categoryWord.filter{ case (category, word) => !stopwords.contains(word) }

// Obtaining the number of reviews per word
// Of course, remove duplicates to not overcount documents
val reviews_per_word = categoryWord.map(x => (x._2, 1)).reduceByKey(_ + _)

// Obtaining the number of reviews per category
val reviews_per_category = categoryReview.map(x => (x._1, 1)).reduceByKey(_ + _)

// Obtaining the total number of reviews (N)
// by summing up the number of reviews per category
val N = reviews_per_category.map(x => x._2).reduce(_ + _)

In [ ]:
// Cleaned RDD
pprint(categoryWord,20)
println()
pprint(reviews_per_word,10)
println()
pprint(reviews_per_category,10)

In [ ]:
/// The fun starts here:

// Computing A
val aValuesRDD: RDD[((String, String), Int)] = categoryWord.map(pair => (pair, 1)).reduceByKey(_ + _)

In [ ]:
pprint(aValuesRDD,10)

In [ ]:
// Compute B values
val bValuesRDD: RDD[((String, String), Int)] = aValuesRDD.map{ case ((category, word), a) => (word, (category, a)) }
    .join(reviews_per_word)
    .map{ case (word, ((category, a), rev_per_word)) => ((category, word), (rev_per_word - a)) }

In [ ]:
pprint(bValuesRDD,10)

In [ ]:
// Compute C values
val cValuesRDD: RDD[((String, String), Int)] = aValuesRDD.map{ case ((category, word), a) => (category, (word, a)) }
    .join(reviews_per_category)
    .map{ case (category, ((word, a), rev_per_cat)) => ((category, word), (rev_per_cat - a)) }

In [ ]:
pprint(cValuesRDD,10)

In [ ]:
// Combine A, B, and C values
val abcValuesRDD: RDD[((String, String), (Int, Int, Int))] = aValuesRDD
    .join(bValuesRDD)
    .join(cValuesRDD)
    .map{ case ((category, word), ((a, b), c)) => ((category, word), (a, b, c)) }

// Compute chi-square values
val chiSquareRDD: RDD[((String, String), Double)] = abcValuesRDD.map{ case ((category, word), (a, b, c)) =>
    ((category, word), chi_square(N, a, b, c))
}

In [ ]:
pprint(chiSquareRDD,10)

In [ ]:
// Group by category, sort by chi and retrieve top K
val K: Int = 75

val topKChiSquareRDD: RDD[(String, List[(String, Double)])] = chiSquareRDD.map{ case ((category, word), chi) => (category, (word, chi)) }
    .groupByKey()
    .mapValues(_.toList.sortBy(-_._2).take(K))

In [ ]:
topKChiSquareRDD.take(10).foreach(println)

In [ ]:
// Check if path exists and delete it if so. 
// Spark creates a new directory each time an output is created
val outputPath = "data/output"
val fs = FileSystem.get(sc.hadoopConfiguration)
val outputDir = new Path(outputPath)
if (fs.exists(outputDir)) fs.delete(outputDir, true)

// Coalesce the RDD into a single partition and save the result to a file
topKChiSquareRDD
    .coalesce(1)
    .map{ case (category, values) => s"$category: ${values.mkString(", ")}" }
    .saveAsTextFile(outputPath)

In [ ]:
// Terminate Spark context
sc.stop()